# Study-Buddy Quiz Agent (notebook version)

This is a notebook version of the course's [Build a Study-Buddy Quiz Agent](../../docs/projects/study-buddy-agent/index.md) project. It generates quiz questions grounded in a specific notes file, then quizzes you on them right here in the notebook, with an LLM judging whether your typed answers are correct.

The real, canonical version of this project is the local script at [`examples/study-buddy-agent/study_buddy.py`](./study_buddy.py) — read that (and the lesson) for the full walkthrough. This notebook mirrors the same `generate_questions()` / `judge_answer()` / `run_quiz()` logic so you can run it with zero local setup, in Colab, Kaggle, or Binder.

**How this notebook differs from the local script:**
- Instead of reading a file from a `notes/` folder, the notes text is embedded directly below as a Python string (one of the same sample files that ships with the local example: `notes/cell-biology.txt`).
- Instead of a `.env` file, your API key is entered with `getpass()` below so it's never saved in the notebook or shown on screen.
- The quiz loop still uses real `input()` calls — this works fine interactively in Colab, Kaggle, and Binder, exactly like it does in a terminal.

This uses [GitHub Models](https://docs.github.com/en/github-models) by default (a free tier, OpenAI-compatible API) — the same choice `study_buddy.py` makes. See the lesson's [Setup section](../../docs/projects/study-buddy-agent/index.md#setup) for how to get a free token, or swap in a different provider's client if you picked one from that table.


## 1. Install dependencies

In [ ]:
!pip install -q openai python-dotenv


## 2. Enter your API key

Paste your free-tier API key when prompted — it's entered with `getpass()` so it's never displayed or stored in this notebook. If you're using GitHub Models, this is your GitHub personal access token (see the lesson's Setup section for how to create one).

In [ ]:
import getpass
import os

os.environ["GITHUB_TOKEN"] = getpass.getpass("Enter your GitHub Models token (or other provider API key): ")


## 3. Set up the client and the notes text

Same client setup as `study_buddy.py`. The notes text below is the real content of [`notes/cell-biology.txt`](./notes/cell-biology.txt) from the local example, embedded directly so this notebook runs standalone with no file upload needed.

In [ ]:
import json

from openai import OpenAI

MODEL = "gpt-4o-mini"  # confirm this still has a free tier before running
NUM_QUESTIONS = 3  # kept small for a quick notebook demo; study_buddy.py defaults to 5

client = OpenAI(
    api_key=os.environ["GITHUB_TOKEN"],
    base_url="https://models.github.ai/inference",
)


In [ ]:
NOTES_TEXT = """\
Cell Biology: The Basics

Every living thing is made of cells, the smallest unit of life. There are
two broad categories: prokaryotic cells (bacteria and archaea), which have
no nucleus and no membrane-bound organelles, and eukaryotic cells (animals,
plants, fungi, protists), which do.

The nucleus stores the cell's DNA and controls gene expression -- deciding
which proteins get made and when. It's surrounded by a double membrane
called the nuclear envelope, with pores that let molecules pass in and out.

Mitochondria are the cell's power plants. They convert nutrients into ATP
(adenosine triphosphate), the molecule cells actually spend as energy
currency, through a process called cellular respiration. Mitochondria have
their own small DNA, separate from the nucleus -- one of the strongest
pieces of evidence for the endosymbiotic theory, which holds that
mitochondria were once free-living bacteria engulfed by an ancestral cell
roughly 1.5 billion years ago.

The cell membrane (plasma membrane) is a double layer of phospholipids that
separates the inside of the cell from the outside world. It's selectively
permeable: small nonpolar molecules like oxygen pass through freely, but
larger or charged molecules need specific protein channels or transporters
to cross.

Plant cells have three extra structures animal cells lack: a rigid cell
wall made of cellulose for structural support, a large central vacuole that
stores water and maintains internal pressure (turgor), and chloroplasts,
which capture sunlight and convert it into chemical energy through
photosynthesis -- the rough inverse of what mitochondria do with that
energy afterward.

Ribosomes, found either floating free in the cytoplasm or attached to the
endoplasmic reticulum, are where proteins actually get built, translating
the instructions carried by messenger RNA into chains of amino acids.
"""

print(NOTES_TEXT[:200] + "...")


## 4. Generate quiz questions grounded in the notes

Same prompt template and parsing logic as `generate_questions()` in `study_buddy.py`: ask the model for a JSON array of `{"question", "expected_answer"}` pairs, grounded strictly in the notes text above. `expected_answer` is kept by the notebook to judge against later — it's never shown to you before you answer.

In [ ]:
GENERATE_PROMPT_TEMPLATE = """You are a study-buddy quiz generator. Read the
study notes below and write exactly {num_questions} quiz questions that can
ONLY be answered correctly by someone who has read THESE SPECIFIC notes --
not generic questions about the general subject. Base every question and
every expected answer strictly on facts stated in the text.

Reply with ONLY a JSON array, no other text, in this exact shape:
[
  {{"question": "...", "expected_answer": "..."}},
  ...
]

Study notes:
{notes_text}
"""


def generate_questions(notes_text: str, num_questions: int = NUM_QUESTIONS) -> list[dict]:
    """Asks the LLM for a list of {"question", "expected_answer"} dicts,
    grounded in notes_text."""
    prompt = GENERATE_PROMPT_TEMPLATE.format(num_questions=num_questions, notes_text=notes_text)
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    raw = response.choices[0].message.content.strip()
    # Models occasionally wrap JSON in a ```json ... ``` fence despite being asked not to.
    raw = raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return json.loads(raw)


questions = generate_questions(NOTES_TEXT)
print(f"Got {len(questions)} questions.")
for q in questions:
    print("-", q["question"])


## 5. Judge answers

Same `judge_answer()` logic as the local script: send the question, the expected answer, and your typed answer to the LLM, and ask it to grade meaning rather than exact wording.

In [ ]:
JUDGE_PROMPT_TEMPLATE = """You are grading a student's quiz answer. Judge
whether the student's answer is correct, partially correct, or incorrect,
compared to the expected answer below -- the student won't phrase it
identically, so judge on meaning, not exact wording.

Question: {question}
Expected answer: {expected_answer}
Student's answer: {student_answer}

Reply with ONLY JSON, no other text, in this exact shape:
{{"verdict": "correct" | "close" | "incorrect", "feedback": "one brief, encouraging sentence"}}
"""


def judge_answer(question: str, expected_answer: str, student_answer: str) -> dict:
    """Asks the LLM to grade one answer. Returns {"verdict": ..., "feedback": ...}."""
    prompt = JUDGE_PROMPT_TEMPLATE.format(
        question=question, expected_answer=expected_answer, student_answer=student_answer
    )
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    raw = response.choices[0].message.content.strip()
    raw = raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return json.loads(raw)


## 6. Run the quiz

This is the same interactive loop as `run_quiz()` in `study_buddy.py` -- it really does call `input()` and wait for you to type an answer. That works fine in Colab, Kaggle, and Binder, just like a terminal. Run the cell, type your answer to each question, and press Enter.

In [ ]:
score = 0
for i, item in enumerate(questions, start=1):
    print(f"\nQuestion {i}/{len(questions)}: {item['question']}")
    student_answer = input("Your answer: ").strip()

    result = judge_answer(item["question"], item["expected_answer"], student_answer)
    verdict = result.get("verdict", "incorrect")
    feedback = result.get("feedback", "")

    if verdict == "correct":
        score += 1
        print(f"✅ Correct! {feedback}")
    elif verdict == "close":
        score += 0.5
        print(f"🟡 Close. {feedback}")
    else:
        print(f"❌ Not quite. {feedback}")
        print(f"   Expected answer: {item['expected_answer']}")

print(f"\nFinal score: {score}/{len(questions)}")


## Next steps

- Try a different notes file: swap `NOTES_TEXT` above for the contents of [`notes/world-war-two-pacific.txt`](./notes/world-war-two-pacific.txt) or [`notes/python-dictionaries.txt`](./notes/python-dictionaries.txt), or paste in your own notes.
- Run the real thing locally: clone the repo and run `uv run python study_buddy.py` from `examples/study-buddy-agent/` — see the [README](./README.md) and the [lesson](../../docs/projects/study-buddy-agent/index.md) for the full walkthrough, including how to scale this up with retrieval if your notes get long.
